In [0]:
# Parameters supplied by the Databricks Job / Airflow.
dbutils.widgets.text("load_mode", "incremental")
dbutils.widgets.text("record_count", "1000")
dbutils.widgets.text("batch_id", "local_test_001")

LOAD_MODE = dbutils.widgets.get("load_mode").strip().lower()
RECORD_COUNT = int(dbutils.widgets.get("record_count"))
BATCH_ID = dbutils.widgets.get("batch_id").strip()

assert LOAD_MODE in {"historical", "incremental"}
assert RECORD_COUNT > 0
assert BATCH_ID != ""

SAFE_BATCH_ID = "".join(
    character if character.isalnum() or character in {"_", "-"} else "_"
    for character in BATCH_ID
)

print("Load mode    :", LOAD_MODE)
print("Record count :", RECORD_COUNT)
print("Batch ID     :", SAFE_BATCH_ID)

### product generation

In [0]:
products = [
    ("P001", 2.0, 8.0),
    ("P002", 3.0, 7.0),
    ("P003", 1.0, 5.0)
]

products_df = spark.createDataFrame(
    products,
    """
    product_id STRING,
    min_temperature DOUBLE,
    max_temperature DOUBLE
    """
)

display(products_df)

### shipment generation

In [0]:
shipments = [
    (
        "SH001", "P001", "V001",
        "Pune", "Mumbai",
        "2026-10-07T08:00:00Z",
        "2026-10-07T12:00:00Z"
    ),
    (
        "SH002", "P002", "V002",
        "Mumbai", "Pune",
        "2026-10-07T08:30:00Z",
        "2026-10-07T13:00:00Z"
    ),
    (
        "SH003", "P003", "V003",
        "Pune", "Bengaluru",
        "2026-10-07T07:00:00Z",
        "2026-10-07T15:00:00Z"
    ),
    (
        "SH004", "P001", "V004",
        "Delhi", "Pune",
        "2026-10-07T06:00:00Z",
        "2026-10-07T14:00:00Z"
    ),
    (
        "SH005", "P002", "V005",
        "Hyderabad", "Chennai",
        "2026-10-07T09:00:00Z",
        "2026-10-07T16:00:00Z"
    )
]

shipments_df = spark.createDataFrame(
    shipments,
    """
    shipment_id STRING,
    product_id STRING,
    vehicle_id STRING,
    origin STRING,
    destination STRING,
    dispatch_time STRING,
    expected_arrival STRING
    """
)

display(shipments_df)


In [0]:
delivery_events = [
    # 30 min late
    (
        "D001",
        "SH001",
        "DELIVERED",
        "2026-10-07T12:30:00Z"
    ),

    # 10 min early
    (
        "D002",
        "SH002",
        "DELIVERED",
        "2026-10-07T12:50:00Z"
    ),

    # SH003 deliberately not delivered yet

    # exactly on time
    (
        "D004",
        "SH004",
        "DELIVERED",
        "2026-10-07T14:00:00Z"
    ),

    # 60 min late
    (
        "D005",
        "SH005",
        "DELIVERED",
        "2026-10-07T17:00:00Z"
    )
]

deliveries_df = spark.createDataFrame(
    delivery_events,
    """
    event_id STRING,
    shipment_id STRING,
    status STRING,
    event_time STRING
    """
)


display(deliveries_df)

In [0]:
import random
from datetime import datetime, timedelta, timezone
from collections import Counter
import hashlib
SEED = 42

TOTAL_SENSOR_RECORDS = RECORD_COUNT

BAD_RECORD_RATE = 0.01
EXCEPTION_RATE = 0.03
BOUNDARY_RATE = 0.01

BAD_TYPE_WEIGHTS = {
    "invalid_temperature": 0.40,
    "duplicate_event_id": 0.30,
    "missing_event_id": 0.20,
    "unknown_shipment": 0.10,
}

INVALID_TEMPS = ["hot", "N/A", "error", "--"]

# Different batch_id gives a different but reproducible batch.
batch_seed = int(
    hashlib.sha256(SAFE_BATCH_ID.encode("utf-8")).hexdigest()[:8],
    16
)

rng = random.Random(SEED + batch_seed)


def parse_utc(value):
    return datetime.fromisoformat(
        value.replace("Z", "+00:00")
    ).astimezone(timezone.utc)


def format_utc(value):
    return value.strftime("%Y-%m-%dT%H:%M:%SZ")


limits = {
    row["product_id"]: (
        float(row["min_temperature"]),
        float(row["max_temperature"]),
    )
    for row in products_df.collect()
}

shipment_rows = shipments_df.collect()

BAD_RECORD_COUNT = round(
    TOTAL_SENSOR_RECORDS * BAD_RECORD_RATE
)

GOOD_RECORD_COUNT = (
    TOTAL_SENSOR_RECORDS - BAD_RECORD_COUNT
)

per_shipment, remainder = divmod(
    GOOD_RECORD_COUNT,
    len(shipment_rows),
)

good_sensor_events = []
temperature_exception_count = 0
boundary_count = 0

for shipment_index, shipment in enumerate(shipment_rows):
    shipment_id = shipment["shipment_id"]
    sensor_id = "S" + shipment_id[2:]

    minimum_temperature, maximum_temperature = limits[
        shipment["product_id"]
    ]

    dispatch_time = parse_utc(
        shipment["dispatch_time"]
    )

    expected_arrival = parse_utc(
        shipment["expected_arrival"]
    )

    reading_count = (
        per_shipment
        + (1 if shipment_index < remainder else 0)
    )

    for reading_index in range(reading_count):
        event_time = datetime.fromtimestamp(
            rng.uniform(
                dispatch_time.timestamp(),
                expected_arrival.timestamp(),
            ),
            tz=timezone.utc,
        )

        probability = rng.random()

        if probability < EXCEPTION_RATE:
            temperature_exception_count += 1

            if rng.random() < 0.5:
                temperature = (
                    maximum_temperature
                    + rng.uniform(0.5, 5.0)
                )
            else:
                temperature = (
                    minimum_temperature
                    - rng.uniform(0.5, 3.0)
                )

        elif probability < EXCEPTION_RATE + BOUNDARY_RATE:
            boundary_count += 1

            temperature = (
                minimum_temperature
                if rng.random() < 0.5
                else maximum_temperature
            )

        else:
            temperature = rng.uniform(
                minimum_temperature,
                maximum_temperature,
            )

        global_index = len(good_sensor_events) + 1

        event_id = (
            f"{SAFE_BATCH_ID}_E{global_index:06d}"
        )

        good_sensor_events.append(
            (
                event_id,
                shipment_id,
                sensor_id,
                format_utc(event_time),
                f"{temperature:.2f}",
            )
        )

bad_sensor_events = []

bad_types = []
for bad_type, weight in BAD_TYPE_WEIGHTS.items():
    bad_types.extend(
        [bad_type] * int(weight * 100)
    )

for bad_index in range(BAD_RECORD_COUNT):
    bad_type = bad_types[
        bad_index % len(bad_types)
    ]

    event_time = (
        datetime.now(timezone.utc)
        + timedelta(seconds=bad_index)
    )

    event_time_string = format_utc(event_time)

    bad_event_id = (
        f"{SAFE_BATCH_ID}_BAD{bad_index + 1:06d}"
    )

    if bad_type == "missing_event_id":
        record = (
            None,
            "SH005",
            "S005",
            event_time_string,
            "5.0",
        )

    elif bad_type == "unknown_shipment":
        record = (
            bad_event_id,
            "SH999",
            "S999",
            event_time_string,
            "5.0",
        )

    elif bad_type == "invalid_temperature":
        record = (
            bad_event_id,
            "SH005",
            "S005",
            event_time_string,
            rng.choice(INVALID_TEMPS),
        )

    else:
        source_record = good_sensor_events[
            bad_index % len(good_sensor_events)
        ]

        record = source_record

    bad_sensor_events.append(record)

sensor_events = (
    good_sensor_events
    + bad_sensor_events
)

rng.shuffle(sensor_events)

assert len(sensor_events) == TOTAL_SENSOR_RECORDS

print("==============================================")
print("SENSOR GENERATION SUMMARY")
print("==============================================")
print("Batch ID               :", SAFE_BATCH_ID)
print("Load mode              :", LOAD_MODE)
print("Total sensor rows      :", len(sensor_events))
print("Good base rows         :", len(good_sensor_events))
print("Controlled bad rows    :", len(bad_sensor_events))
print("Temperature exceptions :", temperature_exception_count)
print("Boundary readings      :", boundary_count)


In [0]:
# ============================================================
# STORAGE PATH CONFIGURATION
# ============================================================

VOLUME_ROOT = "/Volumes/training_lab_classic/cold_chain/volume1"

LANDING_ROOT = f"{VOLUME_ROOT}/landing"
STAGING_ROOT = f"{VOLUME_ROOT}/staging"

PRODUCTS_PATH = f"{LANDING_ROOT}/product_limits"
SHIPMENTS_PATH = f"{LANDING_ROOT}/shipments"
DELIVERIES_PATH = f"{LANDING_ROOT}/delivery_events"

# Each run gets its own directory.
# This prevents one incremental batch from deleting another.
SENSOR_STAGING = (
    f"{STAGING_ROOT}/sensor_events/"
    f"load_mode={LOAD_MODE}/"
    f"batch_id={SAFE_BATCH_ID}"
)

required_directories = [
    LANDING_ROOT,
    STAGING_ROOT,
    PRODUCTS_PATH,
    SHIPMENTS_PATH,
    DELIVERIES_PATH,
    SENSOR_STAGING,
]

for directory in required_directories:
    dbutils.fs.mkdirs(directory)

print("Storage paths configured")
print("Products path :", PRODUCTS_PATH)
print("Shipments path:", SHIPMENTS_PATH)
print("Deliveries    :", DELIVERIES_PATH)
print("Sensor staging:", SENSOR_STAGING)

### batch simulation

In [0]:
# ============================================================
# SENSOR BATCH CREATION
# ============================================================

BATCH_SIZE = 1000

assert BATCH_SIZE > 0
assert len(sensor_events) == TOTAL_SENSOR_RECORDS

sensor_batches = [
    sensor_events[start:start + BATCH_SIZE]
    for start in range(
        0,
        len(sensor_events),
        BATCH_SIZE,
    )
]

total_batched_records = sum(
    len(batch)
    for batch in sensor_batches
)

assert total_batched_records == TOTAL_SENSOR_RECORDS, (
    "Batching lost or duplicated sensor records"
)

print("==============================================")
print("SENSOR BATCH SUMMARY")
print("==============================================")
print("Total records :", TOTAL_SENSOR_RECORDS)
print("Batch size    :", BATCH_SIZE)
print("Batch count   :", len(sensor_batches))

for batch_number, batch in enumerate(
    sensor_batches,
    start=1,
):
    print(
        f"Batch {batch_number:03d}: "
        f"{len(batch)} records"
    )

### writing in the json format

In [0]:
# ============================================================
# JSON BATCH WRITER
# ============================================================

import json


def write_sensor_batch(filename, records):
    if not filename.endswith(".json"):
        raise ValueError(
            f"Filename must end with .json: {filename}"
        )

    file_path = (
        f"{SENSOR_STAGING.rstrip('/')}/"
        f"{filename}"
    )

    json_lines = []

    for row in records:
        record = {
            "event_id": row[0],
            "shipment_id": row[1],
            "sensor_id": row[2],
            "event_time": row[3],
            "temperature_c": row[4],
            "batch_id": SAFE_BATCH_ID,
            "load_mode": LOAD_MODE,
        }

        json_lines.append(
            json.dumps(
                record,
                ensure_ascii=False,
            )
        )

    file_content = "\n".join(json_lines) + "\n"

    dbutils.fs.put(
        file_path,
        file_content,
        overwrite=True,
    )

    print(
        f"Created {filename} "
        f"with {len(records)} records"
    )

    return file_path


print("JSON batch writer ready")

In [0]:
# Writes the product reference dataset into the landing directory.

(
    products_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(PRODUCTS_PATH)
)

product_files = dbutils.fs.ls(PRODUCTS_PATH)

print("Product data written successfully")
print("Product record count:", products_df.count())

display(product_files)

In [0]:
# Writes the shipment master dataset into the landing directory.

(
    shipments_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(SHIPMENTS_PATH)
)

shipment_files = dbutils.fs.ls(SHIPMENTS_PATH)

print("Shipment data written successfully")
print("Shipment record count:", shipments_df.count())

display(shipment_files)

In [0]:
# Writes the delivery-event dataset as JSON into the landing directory.

(
    deliveries_df
    .write
    .mode("overwrite")
    .json(DELIVERIES_PATH)
)

delivery_files = dbutils.fs.ls(DELIVERIES_PATH)

print("Delivery data written successfully")
print("Delivery event count:", deliveries_df.count())

display(delivery_files)

In [0]:
# ============================================================
# WRITE SENSOR BATCH FILES
# ============================================================

generated_filenames = []
generated_file_paths = []

for batch_number, batch in enumerate(
    sensor_batches,
    start=1,
):
    filename = (
        f"{LOAD_MODE}_"
        f"{SAFE_BATCH_ID}_"
        f"{batch_number:03d}.json"
    )

    file_path = write_sensor_batch(
        filename=filename,
        records=batch,
    )

    generated_filenames.append(filename)
    generated_file_paths.append(file_path)


staging_files = dbutils.fs.ls(SENSOR_STAGING)

generated_file_set = set(generated_filenames)

generated_files = [
    file_info
    for file_info in staging_files
    if file_info.name in generated_file_set
]


assert len(generated_files) == len(sensor_batches), (
    "Generated sensor file count does not match batch count"
)

assert sum(
    len(batch)
    for batch in sensor_batches
) == TOTAL_SENSOR_RECORDS, (
    "Sensor record count mismatch"
)


print("==============================================")
print("DATA GENERATION STATUS: PASS")
print("==============================================")
print("Batch ID           :", SAFE_BATCH_ID)
print("Load mode          :", LOAD_MODE)
print("Sensor records     :", TOTAL_SENSOR_RECORDS)
print("Expected files     :", len(sensor_batches))
print("Generated files    :", len(generated_files))
print("Sensor staging path:", SENSOR_STAGING)

display(generated_files)

In [0]:
print("Sensor source path:", SENSOR_STAGING)

display(dbutils.fs.ls(SENSOR_STAGING))

generated_sensor_df = (
    spark.read
    .json(generated_file_paths)
)

actual_count = generated_sensor_df.count()

print("Expected rows:", TOTAL_SENSOR_RECORDS)
print("Saved rows   :", actual_count)

assert actual_count == TOTAL_SENSOR_RECORDS, (
    f"Row mismatch: expected {TOTAL_SENSOR_RECORDS}, "
    f"found {actual_count}"
)

generated_sensor_df.printSchema()

display(generated_sensor_df.limit(10))